# Custom Model Generation & Synthetic Data

Author: Rohan Dahale

This notebook is **guide** for the `ehteval` framework model and synthetic data generation.  
Use it to:
- Build custom source models
- Generate synthetic EHT observations from those models
- Apply pre-imaging calibration (preimcal) to the synthetic data
- Visualize and inspect the results

> For bulk generation of all models / all datasets, use `generate_models.py` and `generate_data.py` instead.

---

## Structure

```
models_config.yaml  -->  generate_models.py  -->  models/*.hdf5
data_config.yaml    -->  generate_data.py    -->  synthetic_data/
                              │
                    src/models.py          <-- base models functions
                    src/syntheticdata.py   <-- core data generation engine
                    src/preimcal.py        <-- pre-imaging calibration
```

**This notebook** shows how to use `src/models.py` and `src/syntheticdata.py` directly for custom, one-off models.

## 0. Setup

Run from the `ehteval/` root directory.  
Activate environment first: `conda activate evaluation`

In [1]:
import os
import sys
import numpy as np
import ehtim as eh
import matplotlib.pyplot as plt
%matplotlib inline

if not os.path.exists('./src/models.py'):
    raise RuntimeError("Run this notebook from the ehteval/ root directory!")

from src import models
from src.syntheticdata import (
    GainConfig, load_obs_ref,
    apply_scattering, make_synthetic_dataset, apply_preimcal,
)

Welcome to eht-imaging! v 1.2.4 



## 1. Load Reference Observation

The reference observation (`obs_ref`) provides the telescope array geometry, time stamps, and frequency metadata that all models are built against.

It is **not** used for the source flux, only for the array configuration.

In [ ]:
# Load April 11, 2017 LO band reference
UVFITS = "./realdata/hops_3601_SGRA_LO_netcal_LMTcal_10s_ALMArot_JCcalDouble_dcal.uvfits"
ARRAY  = "./arrays/EHT2017.txt"

obs_ref = load_obs_ref(UVFITS, ARRAY)
print(f"Loaded observation: {obs_ref.tstart:.3f}h – {obs_ref.tstop:.3f}h UT")
print(f"Stations: {list(obs_ref.tarr['site'])}")

# Base parameters shared across models
base_params = {
    'fov_uas':        200,
    'npix':           200,
    'total_flux':     2.7,    # Jy — SgrA* flux
    'besttime_start': 10.85,
    'besttime_stop':  14.05,
    'linpol_frac':    0.2,    # fractional linear polarization <m> not m_net
    'circpol_frac':   0.002,  # fractional circular polarization
}

obs_ref = obs_ref.flag_UT_range(UT_start_hour=base_params['besttime_start'], UT_stop_hour=base_params['besttime_stop'], output='flagged')
print(f"Using best-time observation data: {obs_ref.tstart:.3f}h – {obs_ref.tstop:.3f}h UT")

---
## 2. Built-in Models from `src/models.py`

The framework provides several ready-to-use model builders.  
Each accepts a `params` dict and returns an `ehtim.Movie`.

### 2a. Static Models

Static models replicate a single frame across all time steps.

In [4]:
# --- Crescent (mring order=1) ---
crescent_params = {
    **base_params,
    'diameter_uas': 52,
    'alpha_uas':    15,
    'beta1_abs':    0.23,
    'PA':           120,
    'QU_shift':     np.pi / 2,    # twisting inward EVPA pattern
}
crescent_movie = models.make_static_movie(models.make_crescent_frame, crescent_params, obs_ref);
print(f"Crescent: {len(crescent_movie.times)} frames, avg flux = {crescent_movie.avg_frame().total_flux():.2f} Jy")

# Display average frame
crescent_movie.avg_frame().display(plotp=True, show=True);

/opt/miniconda3/envs/evaluation/lib/python3.9/site-packages/ehtim/obsdata.py:509: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(datalist)


Splitting Observation File into 214 times


Building static movie:   0%|          | 0/214 [00:00<?, ?it/s]

Building static movie: 100%|██████████| 214/214 [00:08<00:00, 24.13it/s]



Merging 214 frames from MJD 51544 10.88 hr to MJD 51544 14.05 hr
Crescent: 214 frames, avg flux = 2.70 Jy


In [5]:
# Other static model examples — uncomment to run

# ring_movie  = models.make_static_movie(models.make_ring_frame,  {**base_params, 'diameter_uas': 52, 'alpha_uas': 15}, obs_ref)
# disk_movie  = models.make_static_movie(models.make_disk_frame,  {**base_params, 'disk_diameter_uas': 70, 'blur_uas': 15}, obs_ref)
# point_movie = models.make_static_movie(models.make_point_frame, {**base_params}, obs_ref)

### 2b. Dynamic Models — mring + hotspot

The most physically motivated model: a static m-ring background with an orbiting Gaussian hotspot.

In [6]:
# mring + CW orbiting hotspot (80 min period)
hs_params = {
    **base_params,
    'hs_flux':        0.3,    # hotspot flux in Jy
    'PA':             120,
    'diameter_uas':   52,
    'alpha_uas':      15,
    'beta1_abs':      0.23,
    'ring_radius_uas': 26,
    'hs_fwhm_uas':    20,
    'QU_shift':       np.pi / 2,
    'period_min':     80,
    'direction':      'CW',
}

hs_movie = models.make_mring_hs_movie(hs_params, obs_ref);
print(f"mring+hs: {len(hs_movie.times)} frames, avg flux = {hs_movie.avg_frame().total_flux():.2f} Jy")

hs_movie.avg_frame().display(plotp=True, show=True);

/opt/miniconda3/envs/evaluation/lib/python3.9/site-packages/ehtim/obsdata.py:509: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(datalist)


Splitting Observation File into 214 times


Building mring+hs movie: 100%|██████████| 214/214 [00:37<00:00,  5.69it/s]



Merging 214 frames from MJD 51544 10.88 hr to MJD 51544 14.05 hr
mring+hs: 214 frames, avg flux = 2.70 Jy


---
## 3. Custom Model — Infalling Shearing Hotspot

This section demonstrates how to build a fully custom model.  
The shearing hotspot is a physically motivated model where a Gaussian blob
follows a Keplerian infalling orbit with differential angular velocity (shear). (Rendered in 2D)

You can use this as a template for any custom dynamics.

In [ ]:
from ehtim.imaging.pol_imager_utils import qimage, uimage

def make_mring_shearing_hotspot(params, obs):
    """Build a movie with a static m-ring and a 2D infalling shearing hotspot."""

    tstart = params['besttime_start']
    tstop  = params['besttime_stop']
    t_frames = np.arange(tstart, tstop + 1.0 / 60.0, 1.0 / 60.0)  # 1 min spacing

    fov          = params['fov_uas'] * eh.RADPERUAS
    npix         = params['npix']
    linpol_frac  = params['linpol_frac']
    circpol_frac = params['circpol_frac']

    # Shear / infall parameters
    flux_hs             = 0.3
    R_start             = 26.0 * eh.RADPERUAS
    duration_to_center  = 12.0   # hours to fall to center
    v_infall            = R_start / duration_to_center
    period_hr           = 80 / 60
    omega_0             = 2 * np.pi / period_hr
    r_softening         = 160.0 * eh.RADPERUAS
    target_fwhm         = 20.0 * eh.RADPERUAS
    sigma               = target_fwhm / 2.355
    width_r             = sigma
    width_phi           = sigma / R_start

    # Pre-compute coordinate grid
    xvals = np.linspace(fov/2, -fov/2, npix)
    yvals = np.linspace(-fov/2,  fov/2, npix)
    xg, yg  = np.meshgrid(xvals, yvals)
    r_grid   = np.sqrt(xg**2 + yg**2)
    phi_grid = np.arctan2(yg, xg)

    im_list = []
    for t in t_frames:
        # A. Background m-ring
        model_obj = eh.model.Model(ra=obs.ra, dec=obs.dec, rf=obs.rf, source=obs.source)
        ring_par = {
            'F0': 2.4,
            'd': 52 * eh.RADPERUAS,
            'alpha': 15 * eh.RADPERUAS,
            'beta_list': [0.23 * np.exp(-1j * np.deg2rad(-120))],
            'x0': 0, 'y0': 0,
        }
        background_im = model_obj.add_thick_mring(**ring_par).make_image(fov, npix)

        xx, yy = np.meshgrid(np.linspace(-1, 1, background_im.xdim),
                              np.linspace(-1, 1, background_im.ydim))
        angles = np.angle(xx + 1j * yy)
        qu_shift = np.pi / 2
        background_im.qvec = qimage(background_im.imvec,
                                     linpol_frac * np.ones(background_im.imvec.shape),
                                     angles.flatten() + qu_shift / 2)
        background_im.uvec = uimage(background_im.imvec,
                                     linpol_frac * np.ones(background_im.imvec.shape),
                                     angles.flatten() - qu_shift / 2)
        background_im.vvec = circpol_frac * background_im.imvec

        # B. Shearing hotspot
        t_evolve   = t - tstart
        current_R  = max(0, R_start - (v_infall * t_evolve))
        base_factor = (R_start  + r_softening) ** 1.5
        grid_factor = (r_grid   + r_softening) ** 1.5
        with np.errstate(divide='ignore'):
            omega_r = omega_0 * (base_factor / grid_factor)
        phi_center = 0.0 - (omega_r * t_evolve)
        dphi = phi_grid - phi_center
        dphi = (dphi + np.pi) % (2 * np.pi) - np.pi
        gauss_r   = np.exp(-(r_grid - current_R)**2 / (2 * width_r**2))
        gauss_phi = np.exp(-dphi**2 / (2 * width_phi**2))
        hs_array  = gauss_r * gauss_phi
        hs_sum = np.sum(hs_array)
        if hs_sum > 0:
            hs_array *= flux_hs / hs_sum

        # C. Combine
        final_im = background_im.copy()
        final_im.imvec += hs_array.flatten()
        final_im.time = t
        im_list.append(final_im)

    print(f"Generated {len(im_list)} shearing hotspot frames.")
    return eh.movie.merge_im_list(im_list, interp='linear')


custom_movie = make_mring_shearing_hotspot(base_params, obs_ref);

Generated 194 shearing hotspot frames.

Merging 194 frames from MJD 51544 10.85 hr to MJD 51544 14.07 hr


## 4. Save Model

`save_model()` writes:
- `<name>.hdf5` — movie frames
- `<name>.fits` — time-averaged frame
- `<name>_params.yaml` — generation parameters (provenance)
- `avg/<name>.png` — average frame plot
- `gifs/<name>_total.gif` — total intensity animation
- `gifs/<name>_lp.gif` — linear polarization animation

Set `obs_ref=None` or `no_plots=True` to skip GIF generation.

In [8]:
output_path = "./models/april11/mring+hs-shearing.hdf5"

models.save_model(
    custom_movie,
    output_path,
    params=base_params,
    obs_ref=obs_ref,
    fps=20,
    no_plots=False,     # set True to skip GIFs
    skip_visvar=True,   # set False to also generate visibility variance plots
);

print(f"Saved model to: {output_path}")

  Generating GIFs ...
Splitting Observation File into 641 times
Data time range: 10.885 - 14.049 h
Using 641 observation times.


/opt/miniconda3/envs/evaluation/lib/python3.9/site-packages/ehtim/obsdata.py:509: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(datalist)


Effective Movie time range: 10.850 - 14.067 h
  -> Rendering Total Intensity GIF (20 FPS)
Generating total GIF in parallel...
Saved ./models/april11/gifs/mring+hs-shearing_total.gif
  -> Rendering Linear Polarization GIF (20 FPS)
Generating lp GIF in parallel...
Saved ./models/april11/gifs/mring+hs-shearing_lp.gif
Saved model to: ./models/april11/mring+hs-shearing.hdf5


---
## 5. Generate Synthetic Observations

Use `make_synthetic_dataset()` to go from a model movie → scattered movie → synthetic uvfits.

The pipeline:
1. Extract the best-time window from the model
2. Apply ISM scattering (diffractive blur + refractive screen)
3. Simulate the observation with gain, phase, and thermal noise from a calibration table
4. Write the output uvfits

### 5a. Using a pre-generated caltable (recommended)

`make_synthetic_dataset()` applies scattering, simulates gain corruptions, and writes a synthetic uvfits. It takes a calibration table directory containing pre-computed gain solutions for the array.

The EHT April 11 caltable is provided at `./caltable_april11/`. For custom epochs or non-standard gain settings, generate your own (see §5b).

In [11]:
CALTABLE_DIR = "./caltable_april11/"   # pre-generated April 11 caltable
OUTPUT_DIR   = "./synthetic_data/april11/uncal/"

uvfits_path = make_synthetic_dataset(
    model_path   = output_path,           # model we just saved
    obs_ref      = obs_ref,
    output_dir   = OUTPUT_DIR,
    caltable_dir = CALTABLE_DIR,
    band         = 'LO',
    tstart       = base_params['besttime_start'],
    tstop        = base_params['besttime_stop'],
    apply_scat   = True,
    rngseed      = 1,     # scattering screen seed
    seed         = 1,     # gain noise seed
    add_th_noise = True,
    ampcal       = False,     # add amp corruptions
    phasecal     = False,     # add phase corruptions
)

print(f"Generated synthetic data: {uvfits_path}")

  Loading model: ./models/april11/mring+hs-shearing.hdf5
  Extracting time window


Extracting window: 100%|██████████| 191/191 [00:00<00:00, 737.38it/s]



Merging 191 frames from MJD 57854 10.85 hr to MJD 57854 14.05 hr
  Applying scattering
Warning!! assuming a constant frame duration, but Movie objects now support unequally spaced frames!


/opt/miniconda3/envs/evaluation/lib/python3.9/site-packages/ehtim/scattering/stochastic_optics.py:115: OptimizeWarning: Unknown solver options: xtol
  self.kzeta = minimize(dipole_Anisotropy, A, method='nelder-mead', options={'xtol': 1e-8, 'disp': False}).x


  Saving scattered movie & avg to ./synthetic_data/april11/uncal/
  Simulating observation

Merging 187 frames from MJD 57854 10.88 hr to MJD 57854 14.05 hr
Producing clean visibilities from movie with nfft FT . . . 


/opt/miniconda3/envs/evaluation/lib/python3.9/site-packages/ehtim/obsdata.py:509: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(datalist)


Adding gain + phase errors to data and applying a priori calibration . . . 
Warning!: in add_noise, some SEFDs are <= 0!
NOT recomputing sigmas, which may result in double systematic noise


/opt/miniconda3/envs/evaluation/lib/python3.9/site-packages/ehtim/obsdata.py:558: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(datalist)


Applying Jones Matrices to data . . . 
Warning!: in add_jones_and_noise, some SEFDs are <= 0!
Resorting to data point sigmas, which may add too much systematic noise!
Adding thermal noise to data . . . 
Building NX table
  [OK] Saved: ./synthetic_data/april11/uncal/mring+hs-shearing_LO_scat_gA_gP_th.uvfits
Generated synthetic data: ./synthetic_data/april11/uncal/mring+hs-shearing_LO_scat_gA_gP_th.uvfits


### 5b. Generating a new caltable

Use `generate_caltable()` with a `GainConfig` object to define per-station gain noise and produce a new calibration table. This is needed for custom epochs or non-default gain settings.

> The `generate_data.py` script automatically builds one shared caltable for all models to keep synthetic noise consistent across the batch.

In [10]:
from src.syntheticdata import generate_caltable, GainConfig

# 1. Define the telescope gain noise characteristics
my_gain_config = GainConfig(
    gain_offset = {
        'AA': 0.000, 'AP': 0.000, 'AZ': 0.000, 'JC': 0.000,
        'LM': 0.000, 'SM': 0.000, 'SP': 0.000, 'SR': 0.0
    },
    gain_sigma  = {
        'AA': 0.016, 'AP': 0.072, 'AZ': 0.059, 'JC': 0.047,
        'LM': 0.108, 'SM': 0.008, 'SP': 0.054, 'SR': 0.0
    },
    dterm_offset = {
        'AA': 0.005, 'AP': 0.005, 'AZ': 0.01,  'LM': 0.01,
        'PV': 0.01,  'SM': 0.005, 'JC': 0.01,  'SP': 0.01,
        'SR': 0.01
    }
)

# 2. Generate the caltable using a reference model
my_caltable_dir = generate_caltable(
    mov          = custom_movie, # Use the shearing hotspot movie we made above
    obs_ref      = obs_ref,
    gain_config  = my_gain_config,
    output_dir   = "./caltable_tutorial/",
    seed         = 42,           # RNG seed for realistic noise
    add_th_noise = True,
    ampcal       = False,        # false = add amplitude gains
    phasecal     = False,        # false = add random phase gains between any two scans
)

print(f"Caltable generated at: {my_caltable_dir}")




Merging 189 frames from MJD 57854 10.88 hr to MJD 57854 14.05 hr
Producing clean visibilities from movie with nfft FT . . . 


/opt/miniconda3/envs/evaluation/lib/python3.9/site-packages/ehtim/obsdata.py:509: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(datalist)


Applying Jones Matrices to data . . . 
Warning!: in add_jones_and_noise, some SEFDs are <= 0!
Resorting to data point sigmas, which may add too much systematic noise!
   Applying gain corruption: ampcal-->False
   Applying atmospheric phase corruption: phasecal-->False
Adding thermal noise to data . . . 
  [OK] Caltable saved to: ./caltable_tutorial/
Caltable generated at: ./caltable_tutorial/


### 5c. Pre-imaging Calibration

`apply_preimcal()` runs the standard EHT pre-imaging pipeline on the synthetic uvfits.

The pipeline order is fixed (see `src/preimcal.py` for full documentation):  
netcal → normalize → LMT → JCMT → tavg → merge → syserr → refnoise → PSD → deblur → rescale → timeflag

In [12]:
import ehtim as eh
import scipy.interpolate as interp

# Build light curve from the scattered movie (for lc-netcal)
scat_hdf5 = uvfits_path.replace('.uvfits', '.hdf5')
mv = eh.movie.load_hdf5(scat_hdf5)
lc_time = mv.times
lc_flux = [im.total_flux() for im in mv.im_list()]
lc_func = interp.UnivariateSpline(lc_time, lc_flux, ext=3, k=4, s=0)

preimcal_params = {
    'do_lc_netcal':  True,
    'lc_function':   lc_func,
    'zbl':           2.7,
    'tint':          60,       # 60s time averaging
    'do_deblurr':    False,    # onsky variant
    'ref_optype':    None,
    'do_timeflag':   True,
    'tstart':        base_params['besttime_start'],
    'tstop':         base_params['besttime_stop'],
}

cal_path = apply_preimcal(
    uvfits_path           = uvfits_path,
    output_dir            = "./synthetic_data/april11/besttime/data/netcal+tavg60s/",
    scattered_movie_path  = scat_hdf5,
    preimcal_params       = preimcal_params,
)

print(f"Calibrated data: {cal_path}")

Loading uvfits:  ./synthetic_data/april11/uncal/mring+hs-shearing_LO_scat_gA_gP_th.uvfits
no IF in uvfits header!
POLREP_UVFITS: circ
Number of uvfits Correlation Products: 4
No stations specified in network cal: defaulting to calibrating all stations!
Using Multiprocessing with 1 Processes
Scan 641/641 : [------------------------------]100%
network_cal time: 148.022481 s
No Calibration  Data for SR !
Pre-imaging processing: Obtaining the light curve from the intra-site baselines
Pre-imaging processing: Normalizing the total flux density
Pre-imaging processing: time-averaging.


/opt/miniconda3/envs/evaluation/lib/python3.9/site-packages/ehtim/obsdata.py:509: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(datalist)


Pre-imaging processing: rescaling the total flux density.
time flagged 2056/2056 visibilities
Building NX table
Calibrated data: ./synthetic_data/april11/besttime/data/netcal+tavg60s/mring+hs-shearing_LO_scat_gA_gP_th_netcal_tavg60_onsky.uvfits


---
## 6. Bulk Generation (script reference)

For generating all models / all datasets at once, use the CLI scripts:

```bash
# Generate all models defined in models_config.yaml
python generate_models.py --config models_config.yaml

# Generate specific models only
python generate_models.py --config models_config.yaml --models mring+hsCW point --no-plots

# Dry run (show what would be generated)
python generate_models.py --config models_config.yaml --dry-run

# Generate synthetic data (all steps)
python generate_data.py --config data_config.yaml

# Generate only uncal data for specific models
python generate_data.py --config data_config.yaml --step uncal --models mring+hsCW

# Run pre-imaging calibration only
python generate_data.py --config data_config.yaml --step preimcal
```

---

## 7. Adding a New Model to the Registry

To add a new model type so it works with `generate_models.py`:

**Step 1** — Implement your frame/movie builder in `src/models.py`:  
```python
def make_my_model_movie(params: dict, obs_ref) -> eh.Movie:
    ...
```

**Step 2** — Register it in `generate_models.py`:  
```python
from src.models import make_my_model_movie
registry['my_model'] = (make_my_model_movie, None)
```

**Step 3** — Add a recipe in `models_config.yaml`:  
```yaml
my_model:
  type: "my_model"
  total_flux: 2.7
  # ... your parameters
```

**Step 4** — Add it to `models.generate` list in the YAML, then run:  
```bash
python generate_models.py --config models_config.yaml --models my_model
```